# Capítulo 3 - Python básico para economistas

Este notebook introduce variables, listas, diccionarios, funciones, bucles y condicionales usando hogares simulados. Los datos son didácticos y no representan evidencia empírica real.

## 1. Preparar rutas relativas

Buscamos la raíz del proyecto con `PROJECT_BRIEF.md`. No usamos rutas absolutas.

In [ ]:
from pathlib import Path
import csv
import statistics


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "PROJECT_BRIEF.md").exists():
            return candidate
    raise FileNotFoundError("No se encontró PROJECT_BRIEF.md en los directorios padres.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUT_TABLES = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_FIGURES = PROJECT_ROOT / "outputs" / "figures"
OUTPUT_REPORTS = PROJECT_ROOT / "outputs" / "reports"

for folder in [DATA_PROCESSED, OUTPUT_TABLES, OUTPUT_FIGURES, OUTPUT_REPORTS]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Raíz del proyecto: {PROJECT_ROOT.name}")
print("Carpetas de salida listas.")

## 2. Variables y tipos de datos

Una variable en Python guarda un valor. Una variable económica, además, debe tener definición, unidad, fuente y periodo.

In [ ]:
umbral_didactico = 300.0
moneda = "unidades monetarias simuladas"
datos_son_simulados = True

print(type(umbral_didactico).__name__, umbral_didactico)
print(type(moneda).__name__, moneda)
print(type(datos_son_simulados).__name__, datos_son_simulados)

## 3. Listas y diccionarios

Usamos una lista de diccionarios: cada diccionario representa un hogar ficticio.

In [ ]:
hogares = [
    {"hogar_id": "H001", "ingreso_mensual": 680.0, "miembros": 2, "horas_trabajadas": 160, "zona": "urbana"},
    {"hogar_id": "H002", "ingreso_mensual": 520.0, "miembros": 4, "horas_trabajadas": 140, "zona": "rural"},
    {"hogar_id": "H003", "ingreso_mensual": 950.0, "miembros": 3, "horas_trabajadas": 180, "zona": "urbana"},
    {"hogar_id": "H004", "ingreso_mensual": 410.0, "miembros": 5, "horas_trabajadas": 120, "zona": "rural"},
    {"hogar_id": "H005", "ingreso_mensual": 760.0, "miembros": 1, "horas_trabajadas": 170, "zona": "urbana"},
    {"hogar_id": "H006", "ingreso_mensual": 610.0, "miembros": 3, "horas_trabajadas": 150, "zona": "rural"},
]

for hogar in hogares:
    print(hogar["hogar_id"], hogar["zona"], hogar["ingreso_mensual"])

## 4. Funciones para indicadores económicos simples

Las funciones hacen explícita la regla de cálculo y reducen repetición.

In [ ]:
def ingreso_per_capita(ingreso_mensual: float, miembros: int) -> float:
    if miembros <= 0:
        raise ValueError("El número de miembros debe ser mayor que cero.")
    return ingreso_mensual / miembros


def ingreso_por_hora(ingreso_mensual: float, horas_trabajadas: float) -> float:
    if horas_trabajadas <= 0:
        return 0.0
    return ingreso_mensual / horas_trabajadas


def clasificar_vulnerabilidad(valor: float, umbral: float) -> str:
    if valor < umbral:
        return "vulnerable_didactico"
    return "no_vulnerable_didactico"


hogares_procesados = []
for hogar in hogares:
    ipc = ingreso_per_capita(hogar["ingreso_mensual"], hogar["miembros"])
    iph = ingreso_por_hora(hogar["ingreso_mensual"], hogar["horas_trabajadas"])
    hogares_procesados.append(
        {
            **hogar,
            "ingreso_per_capita": round(ipc, 2),
            "ingreso_por_hora": round(iph, 2),
            "clasificacion": clasificar_vulnerabilidad(ipc, umbral_didactico),
        }
    )

hogares_procesados

## 5. Resumen y exportación de tablas

Exportamos una base procesada y una tabla resumen. Ambas pueden regenerarse ejecutando el notebook completo.

In [ ]:
fieldnames = list(hogares_procesados[0].keys())
processed_path = DATA_PROCESSED / "chapter_03_hogares_simulados.csv"
with processed_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(hogares_procesados)

ingresos_pc = [hogar["ingreso_per_capita"] for hogar in hogares_procesados]
vulnerables = [hogar for hogar in hogares_procesados if hogar["clasificacion"] == "vulnerable_didactico"]

resumen = [
    {"indicador": "hogares_simulados", "valor": len(hogares_procesados)},
    {"indicador": "ingreso_per_capita_promedio", "valor": round(statistics.mean(ingresos_pc), 2)},
    {"indicador": "ingreso_per_capita_mediano", "valor": round(statistics.median(ingresos_pc), 2)},
    {"indicador": "hogares_vulnerables_didacticos", "valor": len(vulnerables)},
    {"indicador": "umbral_didactico", "valor": umbral_didactico},
]

summary_path = OUTPUT_TABLES / "chapter_03_resumen_hogares.csv"
with summary_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["indicador", "valor"])
    writer.writeheader()
    writer.writerows(resumen)

print(f"Base procesada: {processed_path.relative_to(PROJECT_ROOT)}")
print(f"Tabla resumen: {summary_path.relative_to(PROJECT_ROOT)}")
resumen

## 6. Figura reproducible sin paquetes externos

La figura se guarda como SVG. Es una salida visual didáctica, no una estadística oficial.

In [ ]:
def svg_bar_chart(rows: list[dict], output_path: Path) -> None:
    width = 820
    height = 420
    margin_left = 90
    margin_bottom = 70
    plot_width = width - margin_left - 40
    plot_height = height - 70 - margin_bottom
    max_value = max(row["ingreso_per_capita"] for row in rows)
    bar_width = plot_width / len(rows) * 0.65
    gap = plot_width / len(rows) * 0.35

    elements = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="white"/>',
        '<text x="410" y="32" text-anchor="middle" font-family="Arial" font-size="22" font-weight="bold">Ingreso per cápita simulado por hogar</text>',
        '<text x="410" y="58" text-anchor="middle" font-family="Arial" font-size="13" fill="#555">Fuente: datos simulados con fines pedagógicos</text>',
        f'<line x1="{margin_left}" y1="{height - margin_bottom}" x2="{width - 35}" y2="{height - margin_bottom}" stroke="#333"/>',
        f'<line x1="{margin_left}" y1="70" x2="{margin_left}" y2="{height - margin_bottom}" stroke="#333"/>',
    ]

    for index, row in enumerate(rows):
        value = row["ingreso_per_capita"]
        x = margin_left + index * (bar_width + gap) + gap / 2
        bar_height = (value / max_value) * plot_height
        y = height - margin_bottom - bar_height
        color = "#2f6f73" if row["clasificacion"] == "no_vulnerable_didactico" else "#c28f2c"
        elements.extend(
            [
                f'<rect x="{x:.1f}" y="{y:.1f}" width="{bar_width:.1f}" height="{bar_height:.1f}" fill="{color}"/>',
                f'<text x="{x + bar_width / 2:.1f}" y="{y - 8:.1f}" text-anchor="middle" font-family="Arial" font-size="12">{value:.0f}</text>',
                f'<text x="{x + bar_width / 2:.1f}" y="{height - 45}" text-anchor="middle" font-family="Arial" font-size="12">{row["hogar_id"]}</text>',
            ]
        )

    elements.extend(
        [
            '<text x="18" y="225" transform="rotate(-90 18 225)" text-anchor="middle" font-family="Arial" font-size="13">Ingreso per cápita</text>',
            '<text x="410" y="405" text-anchor="middle" font-family="Arial" font-size="12" fill="#555">Nota: clasificación y valores son didácticos; no equivalen a medición oficial.</text>',
            '</svg>',
        ]
    )
    output_path.write_text("\n".join(elements), encoding="utf-8")


figure_path = OUTPUT_FIGURES / "chapter_03_ingreso_per_capita.svg"
svg_bar_chart(hogares_procesados, figure_path)
print(f"Figura: {figure_path.relative_to(PROJECT_ROOT)}")

## 7. Nota metodológica

Todo ejercicio con datos simulados debe declarar sus límites.

In [ ]:
report_path = OUTPUT_REPORTS / "chapter_03_nota_metodologica.md"
report_path.write_text(
    "# Nota metodológica - Capítulo 3\n\n"
    "Los datos de hogares usados en este notebook son simulados y fueron creados "
    "solo para enseñar estructuras básicas de Python: variables, listas, "
    "diccionarios, funciones, bucles y condicionales.\n\n"
    "La clasificación `vulnerable_didactico` no es una medición oficial de pobreza, "
    "no usa una línea institucional, no aplica ponderadores y no describe una "
    "población real.\n\n"
    "Regla de reproducibilidad: no se modificó `data/raw`; los derivados se "
    "guardaron en `data/processed` y `outputs`.\n",
    encoding="utf-8",
)

print(f"Reporte: {report_path.relative_to(PROJECT_ROOT)}")

## 8. Interpretación

El ejercicio muestra cómo convertir datos simulados en indicadores reproducibles. No permite concluir nada sobre hogares reales. Esa prudencia es parte del oficio técnico del economista computacional.